In [ ]:
import os

os.environ["RAY_DEDUP_LOGS"] = "0"
os.environ["RAY_DISABLE_METRICS"] = "1"
os.environ["RAY_DISABLE_METRICS_EXPORT"] = "1"

os.environ["RAY_memory_monitor_refresh_ms"] = "0"
os.environ["RAY_memory_usage_threshold"] = "0.999"

In [ ]:
import holoviews as hv
import hvplot.polars  # noqa: F401
import polars as pl
from sklearn.metrics import accuracy_score

from autotsc import utils
from autotsc.models import *
from autotsc.utils import *
from autotsc.utils import load_dataset

In [ ]:
X_train, y_train, X_test, y_test = load_dataset("CricketZ")
# X_train, y_train, X_test, y_test = load_dataset("MelbournePedestrian")
# X_train, y_train, X_test, y_test = load_dataset("Worms")
# X_train, y_train, X_test, y_test = load_dataset("PhalangesOutlinesCorrect")
X_train, y_train, X_test, y_test = load_dataset("Lightning2")
# X_train, y_train, X_test, y_test = load_dataset("CricketY")
# X_train, y_train, X_test, y_test = load_dataset("OSULeaf")
# X_train, y_train, X_test, y_test = load_dataset("Wine")
X_train, y_train, X_test, y_test = load_dataset("FiftyWords")
# X_train, y_train, X_test, y_test = load_dataset("Haptics")


# X_train, y_train, X_test, y_test = load_dataset("InlineSkate")
X_train, y_train, X_test, y_test = load_dataset("ProximalPhalanxOutlineAgeGroup") # < catch22 best on val here for some reason, bad on test
X_train, y_train, X_test, y_test = load_dataset("Meat")



In [ ]:
#

In [ ]:
with utils.ray_init_or_reuse(num_cpus=24, resources={"meta": 100}, ignore_reinit_error=True):
    model = AutoTSCModel(verbose=1, n_jobs=-1, n_gpus=-1)#, model_selection="fast")
    model.fit(X_train, y_train)

In [ ]:
with utils.ray_init_or_reuse(num_cpus=24, resources={"meta": 100}, ignore_reinit_error=True):
    pred = model.predict(X_test)
    print("----- CA", accuracy_score(y_test, pred))

In [ ]:
model2 = MultiRocketClassifier(n_jobs=-1)
model2.fit(X_train, y_train)

In [ ]:
pred2 = model2.predict(X_test)
print("----- CA2", accuracy_score(y_test, pred2))

In [ ]:
from aeon.classification.hybrid import HIVECOTEV2

model2 = HIVECOTEV2(n_jobs=-1, time_limit_in_minutes=2)
model2.fit(X_train, y_train)

In [ ]:
pred2 = model2.predict(X_test)
print("----- CA3", accuracy_score(y_test, pred2))

In [ ]:
summary = model.summary()
summary

In [ ]:
model.oof_predictions_

In [ ]:
with utils.ray_init_or_reuse(num_cpus=24, resources={"meta": 100}, ignore_reinit_error=True):
    test_accs = []
    preds = model.predict_per_model(X_test)
    for m, p in preds.items():
        acc = accuracy_score(y_test, p)
        test_accs.append(
            {
                "model_id": m,
                "test_accuracy": acc,
            }
        )
    test_accs = pl.DataFrame(test_accs)
    summary = summary.join(test_accs, on="model_id")

In [ ]:
pl.Config.set_fmt_str_lengths(500)
for r in summary.sort("validation_accuracy").iter_rows():
    print(r)

In [ ]:
# Create the scatter plot

scatter = summary.hvplot.scatter(
    x="validation_accuracy",
    y="test_accuracy",
    c="stacking_level",
    hover_cols=["model_id", "validation_accuracy", "test_accuracy", "stacking_level"],
    cmap="viridis",
    size=80,
    alpha=0.7,
    title="Validation vs Test Accuracy per Model",
    xlabel="Validation Accuracy",
    ylabel="Test Accuracy",
    width=700,
    height=600,
    grid=True,  # Add grid here
)

# Add text labels for model_id
labels = hv.Labels(
    summary.select(["validation_accuracy", "test_accuracy", "model_id"]).to_pandas(),
    kdims=["validation_accuracy", "test_accuracy"],
    vdims=["model_id"],
).opts(
    text_font_size="8pt",
    text_align="left",
    text_baseline="bottom",
    xoffset=0.0005,  # slight offset so text doesn't overlap point
    yoffset=0.0005,
)

# Add diagonal line
min_val = min(summary["validation_accuracy"].min(), summary["test_accuracy"].min())
max_val = max(summary["validation_accuracy"].max(), summary["test_accuracy"].max())
diagonal = hv.Curve([(min_val, min_val), (max_val, max_val)]).opts(
    color="gray", line_dash="dashed", line_width=1
)

# Combine all
plot = scatter * labels * diagonal
plot